In [1]:
from dance.transforms import Compose
from dance.datasets.spatial import SpatialLIBDDataset
from dance.modules.spatial.spatial_domain.spagcn import SpaGCN
from dance.utils import set_seed
from dance.transforms import CellPCA, Compose, FilterGenesMatch, SetConfig
from dance.transforms.graph import SpaGCNGraph, SpaGCNGraph2D

/mnt/g/Projects/dance/dance/utils/matrix.py:178: NumbaExperimentalFeatureWarning: First-class function type feature is experimental
  for j in numba.prange(n):


In [2]:
import scanpy as sc
import numpy as np
import gc
import pandas as pd

In [3]:
from dance.modules.spatial.spatial_domain.stagate import Stagate

In [4]:
import sklearn.metrics

In [5]:
from tqdm import tqdm

In [6]:
from dance import logger
logger.setLevel("ERROR")

In [7]:
import time


In [8]:
from dance.transforms import AnnDataTransform, Compose, SetConfig
from dance.transforms.graph import StagateGraph
from dance.typing import Any, LogLevel, Optional, Tuple

def preprocessing_pipeline(model_name: str = "knn", radius: float = 150, n_neighbors: int = 8, 
                           log_level: LogLevel = "ERROR"):
        return Compose(
            AnnDataTransform(sc.pp.normalize_total, target_sum=1e4),
            AnnDataTransform(sc.pp.log1p),
            StagateGraph(model_name, radius=radius, n_neighbors=n_neighbors),
            SetConfig({
                "feature_channel": "StagateGraph",
                "feature_channel_type": "obsp",
                "label_channel": "label",
                "label_channel_type": "obs"
            }),
            log_level=log_level,
        )

preprocessing_pipeline = preprocessing_pipeline()

In [9]:
from dance.datasets.base import Data
aris = []
nmis = []
t0 = time.time()
adata = sc.read_h5ad("../../../data/slidetags/HumanTonsil_2000.h5ad")
pixel_factor = 80 / ((adata.obsm['spatial'][:, 0].max() - adata.obsm['spatial'][:, 0].min()) * 
       (adata.obsm['spatial'][:, 1].max() - adata.obsm['spatial'][:, 1].min()) 
       / adata.shape[0]) ** .5
adata.obsm['spatial_pixel'] = adata.obsm['spatial'] * pixel_factor 
adata.obs['label'] = 0

data = Data(adata)
preprocessing_pipeline(data)

adj, y = data.get_data(return_type="default")
x = data.data.X
edge_list_array = np.vstack(np.nonzero(adj))

model = Stagate(hidden_dims=[x.shape[1], 512, 32])
pred = model.fit_predict((x, edge_list_array), 1000, num_cluster=6, random_state=0)
adata.obs['stagate'] = pred
adata.obs[['stagate']].to_csv(f"../../Steamboat/revised/saved_results/tonsil_stagate_fine_spatial_domain.csv")

# ari = sklearn.metrics.adjusted_rand_score(adata.obs['parcellation_division'], adata.obs['stagate'])
# nmi = sklearn.metrics.normalized_mutual_info_score(adata.obs['parcellation_division'], adata.obs['stagate'])

# print(i, adata.obs['stagate'].nunique(), ari, nmi, f"{(time.time() - t0) / 60:.2f} minutes")
# aris.append(ari)
# nmis.append(nmi)

# del adata
# gc.collect()

tt = time.time() - t0

100%|████████████████████████████████████| 1000/1000 [00:23<00:00, 43.35it/s]


In [10]:
adata.obs['stagate'].value_counts()

stagate
2    1341
1    1340
4     966
3     851
5     721
0     559
Name: count, dtype: int64

In [12]:
import pandas as pd

df = pd.crosstab(adata.obs['stagate'], adata.obs['cluster'])
df / df.sum(axis=0)

cluster,B_germinal_center,B_memory,B_naive,FDC,NK,T_CD4,T_CD8,T_double_neg,T_follicular_helper,mDC,myeloid,pDC,plasma
stagate,,,,,,,,,,,,,
0,0.155303,0.005814,0.009372,0.094118,0.041176,0.014121,0.008403,0.021739,0.595238,0.000000,0.068627,0.000000,0.122137
1,0.017316,0.604651,0.748828,0.172549,0.105882,0.047497,0.159664,0.086957,0.054422,0.087591,0.068627,0.031250,0.072519
2,0.052489,0.317829,0.207123,0.282353,0.447059,0.381258,0.584034,0.500000,0.122449,0.613139,0.401961,0.718750,0.171756
3,0.400974,0.009690,0.004686,0.113725,0.017647,0.006418,0.025210,0.021739,0.054422,0.000000,0.166667,0.015625,0.083969
4,0.361472,0.003876,0.003749,0.270588,0.017647,0.008986,0.025210,0.000000,0.153061,0.000000,0.205882,0.015625,0.534351
5,0.012446,0.058140,0.026242,0.066667,0.370588,0.541720,0.197479,0.369565,0.020408,0.299270,0.088235,0.218750,0.015267
